In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('cleaned_data.csv')

print("PART 3 SETUP")
print("=" * 50)
print("Dataset loaded successfully")
print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

PART 3 SETUP
Dataset loaded successfully
Dataset shape: (1460, 81)

First five rows:


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [17]:
# Create feature matrix and target labels

X = df.drop(columns=['SalePrice'])

y_reg = df['SalePrice']

y_clf = (y_reg > y_reg.median()).astype(int)


# Identify categorical columns

categorical_columns = X.select_dtypes(include=['object']).columns.tolist()


# Ordinal columns with natural order

ordinal_columns = [
    'ExterQual',
    'ExterCond',
    'HeatingQC',
    'KitchenQual'
]

ordinal_mapping = {
    'Po': 0,
    'Fa': 1,
    'TA': 2,
    'Gd': 3,
    'Ex': 4
}


# Encode ordinal columns

X_encoded = X.copy()

for column in ordinal_columns:
    X_encoded[column] = X_encoded[column].map(ordinal_mapping)


# One-hot encode nominal categorical columns

nominal_columns = [
    column for column in categorical_columns
    if column not in ordinal_columns
]

X_encoded = pd.get_dummies(
    X_encoded,
    columns=nominal_columns,
    drop_first=True
)


# Fill any remaining missing values

X_encoded = X_encoded.fillna(X_encoded.median())


print("PART 3 SETUP")
print("=" * 50)

print("\nOriginal X shape:", X.shape)
print("Encoded X shape:", X_encoded.shape)

print("\nClassification label counts:")
print(y_clf.value_counts().sort_index())

print("\nTotal missing values in X_encoded:",
      X_encoded.isnull().sum().sum())

PART 3 SETUP

Original X shape: (1460, 80)
Encoded X shape: (1460, 235)

Classification label counts:
SalePrice
0    732
1    728
Name: count, dtype: int64

Total missing values in X_encoded: 0


In [18]:
# Split features and classification labels

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X_encoded,
    y_clf,
    test_size=0.2,
    random_state=42
)


# Fit scaler only on training features

scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


print("PART 3 SETUP COMPLETE")
print("=" * 50)

print("\nX_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

print("\ny_clf_train shape:", y_clf_train.shape)
print("y_clf_test shape:", y_clf_test.shape)

print("\nSetup complete. Ready for Task 1.")

PART 3 SETUP COMPLETE

X_train_scaled shape: (1168, 235)
X_test_scaled shape: (292, 235)

y_clf_train shape: (1168,)
y_clf_test shape: (292,)

Setup complete. Ready for Task 1.


TASK 1: DECISION TREE BASELINE

In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Train unconstrained Decision Tree
decision_tree_baseline = DecisionTreeClassifier(
    random_state=42
)

decision_tree_baseline.fit(
    X_train_scaled,
    y_clf_train
)

# Generate predictions
y_train_pred_tree = decision_tree_baseline.predict(X_train_scaled)
y_test_pred_tree = decision_tree_baseline.predict(X_test_scaled)

# Calculate accuracies
train_accuracy_tree = accuracy_score(
    y_clf_train,
    y_train_pred_tree
)

test_accuracy_tree = accuracy_score(
    y_clf_test,
    y_test_pred_tree
)

print("TASK 1: DECISION TREE BASELINE")
print("=" * 50)

print("\nTraining Accuracy:", train_accuracy_tree)
print("Test Accuracy:", test_accuracy_tree)
print("Train-Test Accuracy Gap:",
      train_accuracy_tree - test_accuracy_tree)

TASK 1: DECISION TREE BASELINE

Training Accuracy: 1.0
Test Accuracy: 0.8972602739726028
Train-Test Accuracy Gap: 0.10273972602739723


TASK 2: CONTROLLED DECISION TREE

In [20]:
decision_tree_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

decision_tree_controlled.fit(X_train_scaled, y_clf_train)

train_accuracy_controlled = decision_tree_controlled.score(
    X_train_scaled, y_clf_train
)

test_accuracy_controlled = decision_tree_controlled.score(
    X_test_scaled, y_clf_test
)

print("TASK 2: CONTROLLED DECISION TREE")
print("=" * 50)

print("\nTraining Accuracy:", train_accuracy_controlled)
print("Test Accuracy:", test_accuracy_controlled)

TASK 2: CONTROLLED DECISION TREE

Training Accuracy: 0.9272260273972602
Test Accuracy: 0.8767123287671232


TASK 3: GINI vs ENTROPY COMPARISON

In [21]:
# Train Decision Tree using Gini criterion

gini_tree = DecisionTreeClassifier(
    max_depth=5,
    criterion='gini',
    random_state=42
)

gini_tree.fit(X_train_scaled, y_clf_train)

gini_test_accuracy = gini_tree.score(
    X_test_scaled,
    y_clf_test
)


# Train Decision Tree using Entropy criterion

entropy_tree = DecisionTreeClassifier(
    max_depth=5,
    criterion='entropy',
    random_state=42
)

entropy_tree.fit(X_train_scaled, y_clf_train)

entropy_test_accuracy = entropy_tree.score(
    X_test_scaled,
    y_clf_test
)


print("TASK 3: GINI VS ENTROPY COMPARISON")
print("=" * 50)

print("\nGini Test Accuracy:", gini_test_accuracy)
print("Entropy Test Accuracy:", entropy_test_accuracy)

TASK 3: GINI VS ENTROPY COMPARISON

Gini Test Accuracy: 0.8938356164383562
Entropy Test Accuracy: 0.9006849315068494


TASK 4: RANDOM FOREST

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Train Random Forest model

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(X_train_scaled, y_clf_train)


# Training and test accuracy

rf_train_accuracy = model.score(
    X_train_scaled,
    y_clf_train
)

rf_test_accuracy = model.score(
    X_test_scaled,
    y_clf_test
)


# Test ROC-AUC

rf_test_probabilities = model.predict_proba(
    X_test_scaled
)[:, 1]

rf_test_auc = roc_auc_score(
    y_clf_test,
    rf_test_probabilities
)


# Get feature importance scores

feature_importance = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Importance": model.feature_importances_
})

top_5_features = feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(5)


print("TASK 4: RANDOM FOREST")
print("=" * 50)

print("\nTraining Accuracy:", rf_train_accuracy)
print("Test Accuracy:", rf_test_accuracy)
print("ROC-AUC:", rf_test_auc)

print("\nTop 5 Features by Importance:")
print(top_5_features.to_string(index=False))

TASK 4: RANDOM FOREST

Training Accuracy: 0.9957191780821918
Test Accuracy: 0.9452054794520548
ROC-AUC: 0.9844483428950737

Top 5 Features by Importance:
    Feature  Importance
  GrLivArea    0.083904
  YearBuilt    0.060458
OverallQual    0.059282
   FullBath    0.051729
TotalBsmtSF    0.037707


4 a: GRADIENT BOOSTING

In [23]:
from sklearn.ensemble import GradientBoostingClassifier

# Train Gradient Boosting model

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_boosting_model.fit(
    X_train_scaled,
    y_clf_train
)


# Training and test accuracy

gb_train_accuracy = gradient_boosting_model.score(
    X_train_scaled,
    y_clf_train
)

gb_test_accuracy = gradient_boosting_model.score(
    X_test_scaled,
    y_clf_test
)


# Test ROC-AUC

gb_test_probabilities = gradient_boosting_model.predict_proba(
    X_test_scaled
)[:, 1]

gb_test_auc = roc_auc_score(
    y_clf_test,
    gb_test_probabilities
)


print("TASK 4a: GRADIENT BOOSTING")
print("=" * 50)

print("\nTraining Accuracy:", gb_train_accuracy)
print("Test Accuracy:", gb_test_accuracy)
print("ROC-AUC:", gb_test_auc)

TASK 4a: GRADIENT BOOSTING

Training Accuracy: 0.9888698630136986
Test Accuracy: 0.9452054794520548
ROC-AUC: 0.9857285097909061


4 b: FEATURE ABLATION STUDY

In [24]:
# Identify the 5 features with the lowest importance scores

lowest_5_features = feature_importance.sort_values(
    by="Importance",
    ascending=True
).head(5)

features_to_remove = lowest_5_features["Feature"].tolist()


# Find their column indices

feature_indices_to_remove = [
    X_encoded.columns.get_loc(feature)
    for feature in features_to_remove
]


# Remove the 5 lowest-importance features

X_train_reduced = np.delete(
    X_train_scaled,
    feature_indices_to_remove,
    axis=1
)

X_test_reduced = np.delete(
    X_test_scaled,
    feature_indices_to_remove,
    axis=1
)


# Train second Random Forest with identical hyperparameters

reduced_random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

reduced_random_forest.fit(
    X_train_reduced,
    y_clf_train
)


# Calculate reduced-model test ROC-AUC

reduced_test_probabilities = reduced_random_forest.predict_proba(
    X_test_reduced
)[:, 1]

reduced_test_auc = roc_auc_score(
    y_clf_test,
    reduced_test_probabilities
)


print("TASK 4b: FEATURE ABLATION STUDY")
print("=" * 50)

print("\n5 Features with Lowest Importance:")
print(lowest_5_features.to_string(index=False))

print("\nFull Model Test ROC-AUC:", rf_test_auc)
print("Reduced Model Test ROC-AUC:", reduced_test_auc)

TASK 4b: FEATURE ABLATION STUDY

5 Features with Lowest Importance:
           Feature  Importance
     LotConfig_FR3         0.0
       Street_Pave         0.0
   Condition2_RRNn         0.0
Exterior1st_CBlock         0.0
   Condition2_RRAe         0.0

Full Model Test ROC-AUC: 0.9844483428950737
Reduced Model Test ROC-AUC: 0.9852069603148261


TASK 5: CROSS- VALIDATED COMPARISON

In [25]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# Define 5-fold stratified cross-validation

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# Define the four required models

logistic_regression = LogisticRegression(
    max_iter=1000
)

controlled_decision_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)


# Models to compare

models = {
    "Logistic Regression": logistic_regression,
    "Controlled Decision Tree": controlled_decision_tree,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model
}


# Perform 5-fold cross-validation

cv_results = []

for model_name, current_model in models.items():

    auc_scores = cross_val_score(
        current_model,
        X_train_scaled,
        y_clf_train,
        cv=cv,
        scoring='roc_auc'
    )

    cv_results.append({
        "Model": model_name,
        "Mean AUC": auc_scores.mean(),
        "Std AUC": auc_scores.std()
    })


# Create comparison table

cv_comparison_table = pd.DataFrame(cv_results)


print("TASK 5: CROSS-VALIDATED COMPARISON")
print("=" * 60)

print("\n5-Fold Cross-Validation ROC-AUC Results:")
print(cv_comparison_table.to_string(index=False))

TASK 5: CROSS-VALIDATED COMPARISON

5-Fold Cross-Validation ROC-AUC Results:
                   Model  Mean AUC  Std AUC
     Logistic Regression  0.956086 0.008220
Controlled Decision Tree  0.924300 0.014311
           Random Forest  0.977065 0.004617
       Gradient Boosting  0.973368 0.008429


TASK 6: HYPERPARAMETER TUNING WITH GridSearchCV

In [26]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

# Define parameter grid

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}


# Build pipeline

pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)


# Define GridSearchCV

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring='roc_auc',
    n_jobs=-1
)


# Fit on unscaled training data

grid_search.fit(
    X_train,
    y_clf_train
)


# Get best results

best_params = grid_search.best_params_
best_score = grid_search.best_score_
best_pipeline = grid_search.best_estimator_


print("TASK 6: HYPERPARAMETER TUNING WITH GRIDSEARCHCV")
print("=" * 70)

print("\nBest Parameters:")
print(best_params)

print("\nBest Cross-Validation ROC-AUC:")
print(best_score)

print("\nTotal Model Configurations Evaluated:", 18)
print("Total Fits Including 5 Folds:", 18 * 5)

TASK 6: HYPERPARAMETER TUNING WITH GRIDSEARCHCV

Best Parameters:
{'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best Cross-Validation ROC-AUC:
0.9782964711437587

Total Model Configurations Evaluated: 18
Total Fits Including 5 Folds: 90


TASK 7: MANUAL LEARNING CURVE

In [27]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

learning_curve_results = []

for fraction in fractions:

    subset_size = int(fraction * len(X_train))

    X_train_subset = X_train.iloc[:subset_size]
    y_train_subset = y_clf_train.iloc[:subset_size]

    best_pipeline.fit(
        X_train_subset,
        y_train_subset
    )

    train_probabilities = best_pipeline.predict_proba(
        X_train_subset
    )[:, 1]

    train_auc = roc_auc_score(
        y_train_subset,
        train_probabilities
    )

    test_probabilities = best_pipeline.predict_proba(
        X_test
    )[:, 1]

    test_auc = roc_auc_score(
        y_clf_test,
        test_probabilities
    )

    learning_curve_results.append({
        "Training fraction": fraction,
        "Training AUC": train_auc,
        "Test AUC": test_auc
    })

learning_curve_table = pd.DataFrame(learning_curve_results)

print("TASK 7: MANUAL LEARNING CURVE")
print("=" * 60)

print("\nManual Learning Curve Results:")
print(learning_curve_table.to_string(index=False))

TASK 7: MANUAL LEARNING CURVE

Manual Learning Curve Results:
 Training fraction  Training AUC  Test AUC
               0.2      1.000000  0.979778
               0.4      1.000000  0.982457
               0.6      0.999992  0.984354
               0.8      0.999986  0.984164
               1.0      0.999941  0.984448


TASK 8: SERIALIZE THE BEST MODEL

In [28]:
import joblib

# Save the best pipeline

joblib.dump(
    best_pipeline,
    'best_model.pkl'
)


# Load the saved model

loaded_model = joblib.load(
    'best_model.pkl'
)


# Predict on two test rows

sample_test_rows = X_test.iloc[:2]

sample_predictions = loaded_model.predict(
    sample_test_rows
)


print("TASK 8: SERIALIZE THE BEST MODEL")
print("=" * 60)

print("\nModel saved successfully as best_model.pkl")
print("Model loaded successfully")

print("\nSample predictions on two test rows:")
print(sample_predictions)

TASK 8: SERIALIZE THE BEST MODEL

Model saved successfully as best_model.pkl
Model loaded successfully

Sample predictions on two test rows:
[0 1]


TASK 9: SUMMARY COMPARISON TABLE

In [29]:
# Fit models before test-set prediction
logistic_regression.fit(X_train_scaled, y_clf_train)
controlled_decision_tree.fit(X_train_scaled, y_clf_train)

# Calculate test-set AUC

logistic_test_auc = roc_auc_score(
    y_clf_test,
    logistic_regression.predict_proba(X_test_scaled)[:, 1]
)

controlled_tree_test_auc = roc_auc_score(
    y_clf_test,
    controlled_decision_tree.predict_proba(X_test_scaled)[:, 1]
)

# Create summary comparison table

summary_table = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Controlled Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "5-Fold CV Mean AUC": cv_comparison_table["Mean AUC"].values,
    "5-Fold CV Std AUC": cv_comparison_table["Std AUC"].values,
    "Test-Set AUC": [
        logistic_test_auc,
        controlled_tree_test_auc,
        rf_test_auc,
        gb_test_auc
    ]
})

print("TASK 9: SUMMARY COMPARISON TABLE")
print("=" * 80)

print("\nFinal Model Comparison:")
print(summary_table.to_string(index=False))

TASK 9: SUMMARY COMPARISON TABLE

Final Model Comparison:
                   Model  5-Fold CV Mean AUC  5-Fold CV Std AUC  Test-Set AUC
     Logistic Regression            0.956086           0.008220      0.966336
Controlled Decision Tree            0.924300           0.014311      0.938410
           Random Forest            0.977065           0.004617      0.984448
       Gradient Boosting            0.973368           0.008429      0.985729
